In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lower, upper, initcap, when, lit,
    current_timestamp, regexp_extract, count
)

# Read from bronze
bronze_df = spark.table("bronze.employees")
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
# cleaning transformatiosn
from pyspark.sql.functions import regexp_extract, to_date

cleaned_df = (
    bronze_df
    # trim whitespace
    .withColumn("name",          trim(col("name")))
    .withColumn("email",         trim(lower(col("email"))))
    .withColumn("seniority",     trim(col("seniority")))
    .withColumn("practice_area", trim(col("practice_area")))
    .withColumn("status",        trim(col("status")))

    #validate email format
    .withColumn("email_valid", when(
        regexp_extract(col("email"), r'^[\w\.\-]+@[\w\.\-]+\.\w+$', 0) != "",
        lit(True)).otherwise(lit(False)))

   
    .withColumn("status", initcap(col("status")))

    # flag nulls
    .withColumn("has_nulls", when(
        col("employee_id").isNull() |
        col("name").isNull() |
        col("email").isNull() |
        col("hire_date").isNull(),
        lit(True)).otherwise(lit(False)))

    # add audit columns
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.employees"))
)

# drop duplicates on employee_id
cleaned_df = cleaned_df.dropDuplicates(["employee_id"])

print(f"Silver row count: {cleaned_df.count()}")
print(f"Invalid emails: {cleaned_df.filter(col('email_valid') == False).count()}")
print(f"Rows with nulls: {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
# writing to silver
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.employees")
)

print(f"Written to silver.employees: {spark.table('silver.employees').count()} rows")